![JupyterLite HIV Drug Resistance Maps: an original conceptual globe, map, and molecular illustration.](assets/hivdr-splash.png)

**[Open this notebook in JupyterLite Lab](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/lab/index.html?path=05_mutation_geographic_clusters.ipynb)** · **[Jupyter Book](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/book/)** · **[Mapping resources](mapping_resources.md)**

*Cover illustration only; its markers and bars are not surveillance data.*

# 05 - Geographic clusters of resistance mutations

This notebook is an analysis template for patient- or sequence-level data. It asks whether a particular mutation is more geographically clustered than expected from the observed sample locations. The packaged 2015 study aggregate does not contain the required patient-level mutation rows, so the notebook reports that limitation rather than manufacturing a result.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

DATA = Path('data')
candidates = [DATA / 'hivdb_patient_mutations.csv', DATA / 'current' / 'hivdb_patient_mutations_current.csv']
denominator_candidates = [DATA / 'hivdr_patient_denominators.csv', DATA / 'current' / 'hivdr_patient_denominators_current.csv']
path = next((p for p in candidates if p.exists()), None)
required = {'patient_id', 'iso3', 'mutation'}
if path is None:
    print('No patient-level mutation file found. Expected one of:', [str(p) for p in candidates])
    print('The 2015 study files support country/class summaries only; no geographic mutation-cluster result is estimable from them.')
    mutations = pd.DataFrame()
    denominators = pd.DataFrame()
else:
    mutations = pd.read_csv(path)
    missing = required - set(mutations.columns)
    if missing:
        raise ValueError(f'{path} is missing required columns: {sorted(missing)}')
    print(f'Loaded {len(mutations):,} mutation observations from {path}')
    denominator_path = next((p for p in denominator_candidates if p.exists()), None)
    denominators = pd.read_csv(denominator_path) if denominator_path else pd.DataFrame()
    if denominators.empty:
        print("No denominator file found; prevalence and Moran's I are not estimable.")
    elif not {'patient_id', 'iso3'}.issubset(denominators.columns):
        raise ValueError('The denominator file requires patient_id and iso3.')

## Mutation prevalence by geography

The preferred input is one row per person-mutation observation, with a stable `patient_id`, `iso3`, `mutation`, and a denominator table containing all sequenced people. Do not calculate prevalence from mutation-positive rows alone.

In [ ]:
if not mutations.empty and not denominators.empty:
    # Deduplicate repeated sequence calls, then attach all tested people.
    observed = mutations[['patient_id', 'iso3', 'mutation']].drop_duplicates()
    counts = (observed.groupby(['iso3', 'mutation'])['patient_id'].nunique()
              .rename('people_with_mutation').reset_index())
    tested = denominators.groupby('iso3')['patient_id'].nunique().rename('people_tested')
    counts = counts.merge(tested, on='iso3', how='left')
    counts['prevalence_pct'] = 100 * counts['people_with_mutation'] / counts['people_tested']
    counts['rank_within_country'] = counts.groupby('iso3')['prevalence_pct'].rank(method='first', ascending=False)
    display(counts.sort_values(['iso3', 'prevalence_pct'], ascending=[True, False]).head(30))
else:
    print('Supply hivdr_patient_denominators.csv to calculate mutation prevalence. Positive rows alone cannot provide a denominator.')

## A simple cluster statistic

For a formal spatial analysis, supply an adjacency or distance-weight matrix and pre-specify the mutation and time window. Moran's I is a screen for spatial autocorrelation, not proof of transmission or causality; sampling density, subtype, study design, and founder effects can create apparent clusters.

In [ ]:
edges_path = DATA / 'country_adjacency.csv'
if not mutations.empty and not denominators.empty and edges_path.exists():
    mutation_to_review = 'M184V'
    edges = pd.read_csv(edges_path)
    edges['weight'] = edges.get('weight', 1.0)
    screen = counts[counts['mutation'].eq(mutation_to_review)].set_index('iso3')['prevalence_pct']
    screen = screen.reindex(sorted(set(edges['iso3_i']) | set(edges['iso3_j']))).dropna()
    z = screen - screen.mean()
    weights = edges[edges['iso3_i'].isin(screen.index) & edges['iso3_j'].isin(screen.index)]
    w = weights['weight'].sum()
    numerator = sum(row.weight * z[row.iso3_i] * z[row.iso3_j] * 2 for row in weights.itertuples())
    morans_i = len(screen) / w * numerator / (z.pow(2).sum()) if w and z.pow(2).sum() else np.nan
    print(f"{mutation_to_review} global Moran's I: {morans_i:.3f} (descriptive screen; no p-value)")
    fig = px.bar(screen.rename('prevalence_pct').reset_index(), x='iso3', y='prevalence_pct', title=f'{mutation_to_review}: prevalence by country')
    fig.show()
else:
    print('Cluster statistic deferred until patient_id, mutation, iso3, and a denominator/weight matrix are supplied.')